# Evolution AI Engine — primera prueba GPU gratuita

Esta notebook es un prototipo separado del generador de producción. Usa Wan 2.1 T2V 1.3B (Apache-2.0), genera dos segmentos cortos y los une en un MP4 H.264. No llama a Magic Hour ni requiere una clave de API comercial.

**Límites de esta prueba:** GPU gratuita sujeta a disponibilidad; se requiere una GPU CUDA con al menos 12 GB de VRAM para este perfil; el resultado inicial es 480p y sin audio. No es todavía un servicio público permanente. Los archivos quedan en `/content/` para descargarlos al final.

## 1. Activa una GPU

En Google Colab selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. Si no hay GPU gratuita disponible, no compres nada: inténtalo más tarde.

In [ ]:
import torch, sys
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU:', p.name)
    print('VRAM (GB):', round(p.total_memory / 1024**3, 2))
else:
    raise RuntimeError('No hay GPU activa. Activa GPU en Entorno de ejecución y vuelve a ejecutar.')


In [ ]:
# Instala Diffusers sin reemplazar el PyTorch/CUDA que ya trae Colab.
%pip -q install -r https://raw.githubusercontent.com/admperdilnet-netizen/Evolution-AI/feature/evolution-ai-engine/engine/worker/requirements.txt
!apt-get -qq update && apt-get -qq install -y ffmpeg


In [ ]:
# Descarga el código de la rama aislada; no toca main ni la web publicada.
!git clone --depth 1 --branch feature/evolution-ai-engine https://github.com/admperdilnet-netizen/Evolution-AI.git /content/Evolution-AI
%cd /content/Evolution-AI
!python engine/worker/wan21_worker.py --config engine/colab/demo_scenes.json --output /content/evolution-engine-test.mp4


In [ ]:
# Verifica el MP4 y descarga el video y su informe técnico.
import json, os
video = '/content/evolution-engine-test.mp4'
report = '/content/evolution-engine-test.report.json'
assert os.path.exists(video) and os.path.getsize(video) > 10000, 'No se encontró un MP4 válido.'
print(json.dumps(json.load(open(report)), indent=2, ensure_ascii=False))
from google.colab import files
files.download(video)
files.download(report)


## Qué demuestra esta primera prueba

Al finalizar verás `PRUEBA COMPLETADA`, la GPU utilizada, los datos de cada escena, la duración y el códec del MP4 unido. La prueba une dos segmentos de aproximadamente dos segundos. Esto demuestra el flujo **modelo abierto → escenas → FFmpeg → MP4**, pero no significa todavía que el motor esté integrado en la web ni que pueda atender usuarios continuamente.